# Profilering av regional elnätsbelastning och avbrott med PROC CHART

## Sammanfattning

En analytiker inom nätdrift använder PROC CHART för att profilera ett syntetiskt urval av mätaravläsningar från matarledningar över fyra servicerregioner och fyra genereringskällor. Denna notebook går igenom stående stapeldiagram, liggande stapeldiagram, blockdiagram och cirkeldiagram för att sammanfatta genereringsmixen, jämföra medel- och totalbelastning per region, blottlägga kvällens efterfrågetopp per timme och rangordna avbrottsminuter per källa — den sortens snabba textbaserade utforskning som ett energi- och försörjningsteam kör innan de satsar på en grafisk rapport. DATA-steget begär 1 200 rader; denna notebook renderades i olicensierat läge, som begränsar arbetsdatasetet till de första 100 avläsningarna, så varje diagram nedan sammanfattar det urvalet på 100 rader.

## Datakällor

| Dataset | Rader | Beskrivning |
|---------|------|-------------|
| `grid_ops` | 100 (syntetiskt urval) | En rad per mätaravläsning på en matarledning i ett regionalt nät, genererad direkt i koden med `call streaminit(20260531)` och `rand()`. DATA-stegets loop begär 1 200 avläsningar, men olicensierat läge begränsar det sparade datasetet till 100 observationer, så diagrammen nedan beskriver de 100 raderna. |

# Profilering av regional elnätsbelastning och avbrott med PROC CHART

PROC CHART producerar teckenbaserade stapel-, block- och cirkeldiagram direkt i utskriften — ingen ODS Graphics-enhet krävs. Det gör den till ett snabbt förstahandsverktyg för utforskning för ett nätdriftsteam som vill *se* formen på sin belastnings- och tillförlitlighetsdata innan de bygger polerade GCHART- eller SGPLOT-visualiseringar.

I denna notebook:

1. Genererar vi en syntetisk månad av mätaravläsningar från matarledningar för en regional nätoperatör.
2. Diagrammerar vi **genereringsmixen** (andel avläsningar per källa).
3. Jämför vi **medel- och total levererad belastning** över serviceregioner.
4. Blottlägger vi **kvällens efterfrågetopp** per timme på dygnet.
5. Använder vi ett **blockdiagram** för att korsa region mot genereringskälla.
6. Rangordnar vi **avbrottsminuter** per källa för att hitta den minst tillförlitliga matningen.

Varje sats och option nedan är standardsyntax för PROC CHART i SAS 9.4.

## Steg 1 — Generera de syntetiska driftdata

DATA-steget nedan tillverkar mätaravläsningar i en loop på 1 200 iterationer. Varje avläsning tilldelas en serviceregion, en genereringskälla (Gas dominerar, med Vind, Sol och Vatten som utgör resten) och en timme på dygnet. Belastningen är högre i kvällsfönstret 17:00–21:00 (och vi flaggar de avläsningarna som topp), och vi drar avbrottsminuter från en Poissonfördelning. `call streaminit` fixerar slumpfröet så att data är reproducerbara.

NOTE i loggen visar det praktiska resultatet: denna körning sker i olicensierat läge, så det sparade datasetet `grid_ops` begränsas till de första 100 av dessa avläsningar (100 rader, 7 kolumner). Varje diagram som följer sammanfattar det urvalet på 100 rader, och varje PROC CHART-logg bekräftar att den läste 100 rader.

In [1]:
/* Syntetisk drift av matarledningar för en regional nätoperatör */
data grid_ops;
    CALL streaminit(20260531);
    LÄNGD region $12 source $9 peak_flag $3;
    FÄLT regions[4] $12 _temporary_
        ('Norr','Söder','Öster','Väster');
    GÖR meter_id = 1 TILL 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        OM u < 0.42 SÅ source = 'Gas';
        ANNARS OM u < 0.70 SÅ source = 'Vind';
        ANNARS OM u < 0.88 SÅ source = 'Sol';
        ANNARS source = 'Vatten';
        hour = floor(24 * rand('uniform'));
        /* regional förskjutning: Norr (r=1) +5, Väster (r=4) +3 */
        base = 18 + 5 * (r = 1) + 3 * (r = 4);
        OM hour >= 17 AND hour <= 21 SÅ GÖR;
            load_mwh = base + 12 + 6 * rand('normal');
            peak_flag = 'Ja';
        SLUT;
        ANNARS GÖR;
            load_mwh = base + 4 * rand('normal');
            peak_flag = 'Nej';
        SLUT;
        OM load_mwh < 0 SÅ load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        UTDATA;
    SLUT;
    TA_BORT r u base;
KÖR;


NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.23 seconds
  cpu   0.23 seconds


## Steg 2 — Genereringsmix

Vilken andel av avläsningarna svarar varje genereringskälla för? Ett stående stapeldiagram med `TYPE=PERCENT` besvarar detta direkt: stapelhöjderna är procentandelen av alla observationer som faller i varje källkategori. Eftersom `source` är en teckenvariabel behandlar PROC CHART dess värden som diskreta kategorier automatiskt.

In [2]:
PROCEDUR chart data=grid_ops;
    VBAR source / type=PERCENT;
    LABEL source="Källa";
KÖR;


Percent of Källa

         |                ******        
         |                ******        
   40.00 +                ******        
         |                ******        
         |                ******        
   30.00 +                ******        
         |        ******  ******        
         |        ******  ******        
   20.00 +        ******  ******        
         |        ******  ******        
         |******  ******  ******        
         |******  ******  ******  ******
   10.00 +******  ******  ******  ******
         |******  ******  ******  ******
         |******  ******  ******  ******
         |                              
         +------------------------------+
          Vatten   Vind    Gas     Sol  
                      Källa





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 3 — Genomsnittlig levererad belastning per region

Nu byter vi från att räkna till att sammanfatta en mätstorhet. Att namnge `load_mwh` i `SUMVAR=` med `TYPE=MEAN` gör stapellängden till den genomsnittliga belastningen per region, och vi begär statistikkolumnerna explicit: `MEAN` skriver ut genomsnittet bredvid varje stapel och `CFREQ` lägger till en kumulativ frekvenskolumn. Norr bär den högsta genomsnittligt levererade belastningen (medel omkring 28,0 MWh), i linje med den regionala förskjutning som byggts in i data, medan Söder är lägst (omkring 19,8 MWh).

Vi skickar även `ASCENDING`, som ordnar staplarna från lägst till högst medel. I utdata löper staplarna nu från kortast till längst — Söder (19,8), Öster (21,7), Väster (24,2) och Norr (28,0) — så rangordningen kan läsas direkt av stapelordningen såväl som av den utskrivna `Mean`-kolumnen.

In [3]:
PROCEDUR chart data=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
    LABEL region="Region" load_mwh="Levererad belastning (MWh)";
KÖR;


Mean of Region

Region                                                  Mean           N     Percent
                                                                                    
  Söde  ****************************                   19.80       19.80       21.00
  Öste  *******************************                21.72       41.52       29.00
 Väste  **********************************             24.17       65.69       23.00
  Norr  ****************************************       28.03       93.72       27.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 4 — Total belastning per region

Här gör `TYPE=SUM` varje stapel till den *totala* levererade belastningen för regionen snarare än genomsnittet, så de högre staplarna markerar de regioner som levererar mest sammanlagd energi över de provtagna avläsningarna. Norr leder återigen på total levererad belastning.

Satsen begär även `SUBGROUP=peak_flag`, vilket delar upp varje stapel efter om avläsningen föll i kvällens toppfönster. Varje stapel segmenteras med en distinkt symbol per undergruppsnivå — `J` för toppavläsningar (`Ja`) och `N` för övriga (`Nej`) — och en förklaring under diagrammet kopplar symbolerna till nivåerna. Segmenten visar hur stor del av varje regions totala belastning som infaller under kvällens topptimmar 17:00–21:00.

In [4]:
PROCEDUR chart data=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
    LABEL region="Region" load_mwh="Levererad belastning (MWh)" peak_flag="Topptimme";
KÖR;


Sum of Region

         |                        JJJJJJ
         |                        JJJJJJ
         |                        JJJJJJ
     600 +                JJJJJJ  JJJJJJ
         |JJJJJJ          JJJJJJ  JJJJJJ
         |JJJJJJ          JJJJJJ  JJJJJJ
         |JJJJJJ          JJJJJJ  JJJJJJ
     400 +NNNNNN  JJJJJJ  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
     200 +NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |NNNNNN  NNNNNN  NNNNNN  NNNNNN
         |                              
         +------------------------------+
          Väste    Söde    Öste    Norr 
                      Region

Symbol  Topptimme
------  ---------
  N     Nej      
  J     Ja       





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 5 — Belastningsformen över dygnet

`hour` är kontinuerlig, så vi fastställer indelningen själva med en explicit `MIDPOINTS=`-lista med 4-timmarscentra (2, 6, 10, 14, 18, 22). Varje stapel visar den genomsnittliga levererade belastningen för avläsningar nära den timmen. Stapeln centrerad på 18 bör sticka ut — det är kvällens efterfrågetopp som vi byggt in i data.

In [5]:
PROCEDUR chart data=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
    LABEL hour="Timme på dygnet" load_mwh="Levererad belastning (MWh)";
KÖR;


Mean of Timme på dygnet

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                      Timme på dygnet





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 6 — Region efter genereringskälla (blockdiagram)

Ett BLOCK-diagram renderar en liten tvåvägstabell som ett rutnät av block. Med `GROUP=source` och `SUMVAR=load_mwh / TYPE=MEAN` korsar diagrammet varje region mot den genereringskälla som betjänar den, med blockhöjd proportionell mot medelbelastningen — ett kompakt sätt att upptäcka vilka kombinationer av region/källa som bär den tyngsta genomsnittliga belastningen.

In [6]:
PROCEDUR chart data=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
    LABEL region="Region" source="Källa" load_mwh="Levererad belastning (MWh)";
KÖR;


Block chart of Mean of Region by Källa

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
  Väste    Söde    Öste    Norr 
             Region





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 7 — Genereringsmix som ett cirkeldiagram

Samma information om källandelar från steg 2, ritad som en cirkel. PIE med `TYPE=PERCENT` dimensionerar varje tårtbit efter dess procentandel av totala avläsningar och skriver ut en förklaring med tårtbitarnas procentandelar vid sidan av figuren.

In [7]:
PROCEDUR chart data=grid_ops;
    PIE source / type=PERCENT;
    LABEL source="Källa";
KÖR;


Pie chart of Källa

           Källa     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
          Vatten       14.00     14.0%  ####
            Vind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
             Sol       13.00     13.0%  OOOO





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Steg 8 — Avbrottsminuter per källa

Slutligen rangordnar vi tillförlitlighet. `SUMVAR=outage_min` med `TYPE=SUM` summerar avbrottsminuter per källa. Vi skickar `DESCENDING` för att försöka flyta den sämst presterande källan till toppen, men som i steg 3 ordnas staplarna inte om — de skrivs ut i kategoriordning (Vatten, Vind, Gas, Sol) och stapelomordning är ännu inte implementerad. Läs rangordningen från den utskrivna `Sum`-kolumnen: Gas står för flest totala avbrottsminuter i detta urval (122), klart före Vind (64), Sol (43) och Vatten (38). Det följer genereringsmixen — Gas betjänar flest avläsningar, så den ackumulerar flest avbrottsminuter totalt.

In [8]:
PROCEDUR chart data=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum FALLANDE;
    LABEL source="Källa" outage_min="Avbrottsminuter";
KÖR;


Sum of Källa

 Källa                                                   Sum        Cum.     Percent
                                                                     Sum            
Vatten  ************                                      38          38       14.00
  Vind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
   Sol  **************                                    43         267       13.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Tolka resultaten

Att läsa diagrammen tillsammans ger driftteamet en snabb situationsbild (över det urval på 100 rader som denna körning fångade):

- **Genereringsmix (steg 2 och 7).** Gas bär den största andelen avläsningar (omkring 45 %), med Vind på andra plats (omkring 28 %) och Vatten och Sol efter (omkring 14 % och 13 %) — det stående stapeldiagrammet och cirkeln berättar samma historia på två sätt, en användbar rimlighetskontroll.
- **Belastning per region (steg 3 och 4).** Medelbelastnings-HBAR:en visar Norr med den högsta genomsnittligt levererade belastningen (medel ~28 MWh) och Söder den lägsta (~20 MWh), i linje med den regionala förskjutning som byggts in i data. SUM-diagrammet bekräftar att Norr även levererar mest total energi.
- **Daglig belastningsform (steg 5).** Mittpunktsstapeln centrerad på timme 18 är klart högst, vilket bekräftar efterfrågetoppen 17:00–21:00 som vi byggt in i data — exakt där ett elbolag skulle fokusera efterfrågeflexibilitet och kapacitetsplanering.
- **Tillförlitlighet (steg 8).** Att summera avbrottsminuter per källa lyfter fram Gas som den största bidragsgivaren till driftstopp i detta urval (122 minuter), det naturliga nästa målet för underhållsöversyn — även om det mestadels speglar att Gas betjänar flest avläsningar.

En not om stapelordning: `ASCENDING` (steg 3) ordnar staplarna som väntat, och `SUBGROUP=` (steg 4) segmenterar varje stapel med en symbol per nivå och skriver ut en förklaring. `DESCENDING` i steg 8 sorterar dock ännu inte om staplarna i denna PROC CHART-implementation — där läses rangordningen från den utskrivna `Sum`-kolumnen.

PROC CHART ger endast teckenutdata, så för styrelseklara visualiseringar skulle teamet flytta dessa samma vyer till PROC GCHART eller PROC SGPLOT. Men som en första genomgång utan installation över ett färskt utdrag besvarar dessa diagram de operativa frågorna — mix, storleksordning och tidpunkt — på sekunder.